In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import re

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense, Dropout

In [2]:
# 1. IMDB 데이터 다운로드
imdb = keras.datasets.imdb

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=10000)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
# 2. 데이터 확인
print("훈련 데이터 개수:", len(x_train))
print("테스트 데이터 개수:", len(x_test))

print("첫 번째 훈련 데이터:")
print(x_train[0])

print("첫 번째 리뷰 길이:", len(x_train[0]))
print("두 번째 리뷰 길이:", len(x_train[1]))

print("첫 번째, 두 번째 레이블:")
print(y_train[0], y_train[1])

print("레이블 개수:")
print(np.unique(y_train, return_counts=True))

훈련 데이터 개수: 25000
테스트 데이터 개수: 25000
첫 번째 훈련 데이터:
[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]
첫 번째 리뷰 길이: 

In [4]:
# 3. 정수 인덱스를 단어로 복원하기 위한 딕셔너리 생성
word_to_index = imdb.get_word_index()

# 처음 몇 개의 인덱스는 특수 용도로 사용
word_to_index = {k: (v + 3) for k, v in word_to_index.items()}
word_to_index["<PAD>"] = 0
word_to_index["<START>"] = 1
word_to_index["<UNK>"] = 2
word_to_index["<UNUSED>"] = 3

index_to_word = dict((value, key) for (key, value) in word_to_index.items())

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [5]:
# 4. 첫 번째 리뷰를 실제 단어로 출력
decoded_review = ' '.join([index_to_word[index] for index in x_train[0]])
print("복원된 첫 번째 리뷰:")
print(decoded_review)

복원된 첫 번째 리뷰:
<START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and now the same being director <UNK> father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for <UNK> and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also <UNK> to the two little boy's that played the <UNK> of norman and paul they were just brilliant children are often left out of the <UNK> list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be pr

In [6]:
# 5. 리뷰 길이를 100으로 통일
x_train = pad_sequences(x_train, maxlen=100)
x_test = pad_sequences(x_test, maxlen=100)

print("패딩 후 첫 번째, 두 번째 리뷰 길이:")
print(len(x_train[0]), len(x_train[1]))
print(len(x_train[0]))

패딩 후 첫 번째, 두 번째 리뷰 길이:
100 100
100


In [7]:
# 6. 신경망 모델 구축
vocab_size = 10000

model = Sequential()
model.add(Embedding(vocab_size, 64, input_length=100))
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
# 7. 모델 컴파일
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [9]:
# 8. 모델 학습
history = model.fit(
    x_train,
    y_train,
    batch_size=64,
    epochs=20,
    verbose=1,
    validation_data=(x_test, y_test)
)

Epoch 1/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.7724 - loss: 0.4541 - val_accuracy: 0.8330 - val_loss: 0.3667
Epoch 2/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9359 - loss: 0.1755 - val_accuracy: 0.8273 - val_loss: 0.4137
Epoch 3/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9923 - loss: 0.0310 - val_accuracy: 0.8308 - val_loss: 0.5489
Epoch 4/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9991 - loss: 0.0062 - val_accuracy: 0.8352 - val_loss: 0.6058
Epoch 5/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9997 - loss: 0.0025 - val_accuracy: 0.8355 - val_loss: 0.6631
Epoch 6/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 1.0000 - loss: 9.7629e-04 - val_accuracy: 0.8376 - val_loss: 0.7028
Epoch 7/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 1.0000 - loss: 6.3976e-04 - val_accuracy: 0.8372 - val_loss: 0.7329
Epoch 8/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 1.0000 - loss: 3.5461e-04 - val_

In [10]:
# 9. 모델 평가
results = model.evaluate(x_test, y_test, verbose=2)
print("테스트 손실과 정확도:")
print(results)

782/782 - 2s - 3ms/step - accuracy: 0.8195 - loss: 1.2678
테스트 손실과 정확도:
[1.2677888870239258, 0.8195199966430664]


In [11]:
# 직접 작성한 영화 리뷰 테스트
review = """What can I say about this movie that was already said?
It is my favorite time travel sci-fi, adventure epic comedy in the 80's
and I love this movie to death! When I saw this movie I was thrown out by its theme.
An excellent sci-fi, adventure epic, I LOVE the 80s.
It's simple the greatest time travel movie ever happened in the history of world cinema.
I love this movie to death, I love, LOVE, love it!"""

In [12]:
# 11. 리뷰 전처리
review = re.sub("[^0-9a-zA-Z ]", "", review).lower()

review_encoding = []

for w in review.split():
    index = word_to_index.get(w, 2)   # 딕셔너리에 없으면 <UNK>

    if index <= 10000:
        review_encoding.append(index)
    else:
        review_encoding.append(word_to_index["<UNK>"])

In [13]:
# 12. 입력 형태 맞추기
test_input = pad_sequences([review_encoding], maxlen=100)

value = model.predict(test_input)

print("예측값:", value)

if value > 0.5:
    print("긍정적인 리뷰입니다.")
else:
    print("부정적인 리뷰입니다.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 356ms/step
예측값: [[1.]]
긍정적인 리뷰입니다.
